In [ ]:
# =========================================================
# ⚙️ CONFIGURATION & SEEDING (STRICT DETERMINISM)
# =========================================================
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pennylane as qml
import optuna

from scipy import stats
from scipy.stats import pearsonr
from math import sqrt
from rdkit import Chem
from rdkit.Chem import MACCSkeys, rdMolDescriptors, AllChem, rdFingerprintGenerator
from lifelines.utils import concordance_index
from Bio.SeqUtils.ProtParam import ProteinAnalysis

from sklearn.preprocessing import MaxAbsScaler, StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer

# Global Seed Initialization
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
tf.keras.utils.set_random_seed(seed)
tf.config.experimental.enable_op_determinism()

# Fix for Kaggle GPU Initialization
physical_devices = tf.config.list_physical_devices('GPU')
if physical_devices:
    try:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
    except RuntimeError as e:
        print(e)

# =========================================================
# 📊 CUSTOM EVALUATION FUNCTIONS (DTA RESEARCH STANDARDS)
# =========================================================

def rmse(y, f):
    return sqrt(((y - f)**2).mean(axis=0))

def mse(y, f):
    return ((y - f)**2).mean(axis=0)

def pearson(y, f):
    return np.corrcoef(y, f)[0, 1]

def spearman(y, f):
    return stats.spearmanr(y, f)[0]

def ci(y, f):
    ind = np.argsort(y)
    y = y[ind]
    f = f[ind]
    i = len(y) - 1
    j = i - 1
    z = 0.0
    S = 0.0
    while i > 0:
        while j >= 0:
            if y[i] > y[j]:
                z = z + 1
                u = f[i] - f[j]
                if u > 0:
                    S = S + 1
                elif u == 0:
                    S = S + 0.5
            j = j - 1
        i = i - 1
        j = i - 1
    return S / z

def get_r2m(y, f):
    r2 = r2_score(y, f)
    r02 = r2_score(y, f, force_finite=False)
    return r2 * (1 - np.sqrt(np.abs(r2 - r02)))

# =========================================================
# 📊 DATA LOADING & PREPROCESSING
# =========================================================
df = pd.read_csv('data/davis.txt', sep=' ', header=None)
df.columns = ['drug', 'protein', 'smiles', 'sequence', 'score']

df = df.drop_duplicates(subset=['smiles', 'sequence'], keep='first')
df = df.dropna()
df = df.reset_index(drop=True)

smiles = df["smiles"].astype(str).values
proteins = df["sequence"].astype(str).values
scores = df["score"].values.astype(np.float32)

# =========================================================
# 🧬 PROTEIN FEATURE EXTRACTION (OPTIMIZED PARAMETERS)
# =========================================================

def extract_biopython_features(sequence_list):
    features = []
    for seq in sequence_list:
        clean_seq = ''.join([aa for aa in seq if aa in "ACDEFGHIKLMNPQRSTVWY"])
        if len(clean_seq) < 5:
            features.append(np.zeros(3)) # Updated to 3 features
            continue
        analysed_seq = ProteinAnalysis(clean_seq)
        
        # Based on Optimization Results: Top 3 Features
        molar_ext = analysed_seq.molar_extinction_coefficient() 
        ext_red = molar_ext[0] # MolarExt_Reduced
        ext_ox = molar_ext[1]  # MolarExt_Oxidized
        mw = analysed_seq.molecular_weight() # Molecular Weight
        
        features.append([ext_red, ext_ox, mw])
    return np.array(features)

X_biopython_raw = extract_biopython_features(proteins)
bp_scaler = StandardScaler()
X_biopython = bp_scaler.fit_transform(X_biopython_raw)

def get_ngrams(sequences, k=4):
    ngram_seqs = [" ".join([seq[i:i+k] for i in range(len(seq) - k + 1)]) for seq in sequences]
    vectorizer = CountVectorizer(max_features=512)
    return vectorizer.fit_transform(ngram_seqs).toarray()

X_ngrams = get_ngrams(proteins, k=4)
ngram_scaler = MaxAbsScaler()
X_ngrams_final = ngram_scaler.fit_transform(X_ngrams)

amino_acids = 'ACDEFGHIKLMNPQRSTVWYX'
aa_to_num = {aa: idx + 1 for idx, aa in enumerate(amino_acids)}

def encode_protein(sequence_list, max_len=500):
    encoded_list = []
    for seq in sequence_list:
        encoded = [aa_to_num.get(aa, 0) for aa in seq]
        encoded = encoded[:max_len] + [0]*(max_len - len(encoded))
        encoded_list.append(encoded)
    return np.array(encoded_list)

X_prot_label = encode_protein(proteins, max_len=500)

# =========================================================
# ⚛️ QUANTUM COMPUTING SETUP (OPTIMIZED WIRES: 2)
# =========================================================
# Updated to 2 wires per Optimization Report
dev_amp = qml.device("default.qubit", wires=2)

@qml.qnode(dev_amp)
def biopython_amplitude_encoding(features):
    # Padding 3 features to 4 to fit 2 qubits (2^2 = 4)
    padded_features = np.pad(features, (0, 1), 'constant')
    qml.AmplitudeEmbedding(padded_features, wires=range(2), normalize=True)
    return [qml.expval(qml.PauliZ(i)) for i in range(2)]

dev_simple = qml.device("default.qubit", wires=2)

@qml.qnode(dev_simple)
def q_feature_amp(x1, x2):
    features = np.array([x1, x2, 0.0, 0.0])
    qml.AmplitudeEmbedding(features, wires=range(2), normalize=True)
    return qml.expval(qml.PauliZ(0))

X_bp_quantum = np.array([biopython_amplitude_encoding(f) for f in X_biopython])
X_biopython_final = np.concatenate([X_biopython, X_bp_quantum], axis=1)

def protein_q_feature(seq):
    f1 = len(seq) / 1000.0
    f2 = len(set(seq)) / len(amino_acids)
    return q_feature_amp(f1, f2)

X_prot_q = np.array([protein_q_feature(seq) for seq in proteins]).reshape(-1,1)
X_prot_q_expanded = np.repeat(X_prot_q, X_prot_label.shape[1], axis=1)
X_prot_final = X_prot_label + X_prot_q_expanded

# =========================================================
# 💊 SMILES FEATURE EXTRACTION
# =========================================================
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=4, fpSize=512)

maccs_fps = []
ecfp_fps = []
for smi in smiles:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        maccs_fps.append(np.zeros(167))
        ecfp_fps.append(np.zeros(512))
    else:
        maccs_fps.append(np.array(MACCSkeys.GenMACCSKeys(mol)))
        ecfp_fps.append(np.array(morgan_gen.GetFingerprint(mol)))

X_maccs = np.array(maccs_fps)
X_ecfp = np.array(ecfp_fps)

smi_chars = sorted(list(set("".join(smiles))))
smi_dict = {c: i + 1 for i, c in enumerate(smi_chars)}

def encode_smiles_chars(smi_list, max_len=100):
    encoded = []
    for s in smi_list:
        tokens = [smi_dict.get(c, 0) for c in s]
        tokens = tokens[:max_len] + [0]*(max_len - len(tokens))
        encoded.append(tokens)
    return np.array(encoded)

X_smi_chars = encode_smiles_chars(smiles, max_len=100)

def smiles_q_feature(smi):
    f1 = len(smi) / 100.0
    special = sum([smi.count(c) for c in ['=', '#', '(', ')']])
    f2 = special / 50.0
    return q_feature_amp(f1, f2)

X_smi_q = np.array([smiles_q_feature(s) for s in smiles]).reshape(-1,1)

X_smi_combined = np.concatenate([X_maccs, X_ecfp, X_smi_chars, X_smi_q], axis=1)
scaler_smi = MaxAbsScaler()
X_smi_final = scaler_smi.fit_transform(X_smi_combined)

# =========================================================
# ✂️ DATA SPLITTING (STRICT 8:1:1 COLD-START RATIO)
# =========================================================
y = scores
total_samples = len(y)
indices = np.arange(total_samples)

# Fixed Seed for Reproducibility
np.random.seed(seed)
np.random.shuffle(indices)

# Calculate Split Sizes for 8:1:1
test_size = int(0.10 * total_samples)
val_size = int(0.10 * total_samples)
train_size = total_samples - test_size - val_size

# Assign Indices
train_idx = indices[:train_size]
val_idx = indices[train_size : train_size + val_size]
test_idx = indices[train_size + val_size:]

# Training Sets
X_smi_train = X_smi_final[train_idx]
X_prot_train = X_prot_final[train_idx]
X_bp_train = X_biopython_final[train_idx]
X_ngram_train = X_ngrams_final[train_idx]
y_train = y[train_idx]

# Validation Sets
X_smi_val = X_smi_final[val_idx]
X_prot_val = X_prot_final[val_idx]
X_bp_val = X_biopython_final[val_idx]
X_ngram_val = X_ngrams_final[val_idx]
y_val = y[val_idx]

# Test Sets (Final Evaluation)
X_smi_test = X_smi_final[test_idx]
X_prot_test = X_prot_final[test_idx]
X_bp_test = X_biopython_final[test_idx]
X_ngram_test = X_ngrams_final[test_idx]
y_test = y[test_idx]

print(f"--- Dataset Split Summary ---")
print(f"Total Samples: {total_samples}")
print(f"Training Set:  {len(train_idx)} ({len(train_idx)/total_samples:.1%})")
print(f"Validation Set: {len(val_idx)} ({len(val_idx)/total_samples:.1%})")
print(f"Test Set:        {len(test_idx)} ({len(test_idx)/total_samples:.1%})")
print(f"-----------------------------")

# =========================================================
# 🏗️ MULTI-GPU INITIALIZATION (KAGGLE T4 x2)
# =========================================================
strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

# =========================================================
# 🏗️ QNN ARCHITECTURE & HYPERPARAMETERS
# =========================================================

LEARNING_RATE = 0.00013068942544389324
DROPOUT_RATE = 0.29038004945599566
BATCH_SIZE = 64
EPOCHS = 415 

def se_block(x):
    se = tf.keras.layers.Dense(x.shape[-1] // 2, activation="relu")(x)
    se = tf.keras.layers.Dense(x.shape[-1], activation="sigmoid")(se)
    return tf.keras.layers.Multiply()([x, se])

def res_block(x, units):
    shortcut = x
    x = tf.keras.layers.Dense(units, activation="relu")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(units, activation="relu")(x)
    if shortcut.shape[-1] != units:
        shortcut = tf.keras.layers.Dense(units)(shortcut)
    return tf.keras.layers.Add()([x, shortcut])

def build_qnn_model(smi_dim, bp_dim, ngram_dim, vocab_size, lr=LEARNING_RATE):
    drug_input = tf.keras.layers.Input(shape=(smi_dim,))
    d = tf.keras.layers.Dense(1024, activation="relu")(drug_input)
    d = tf.keras.layers.BatchNormalization()(d)
    d = tf.keras.layers.Dropout(DROPOUT_RATE)(d) 
    d = tf.keras.layers.Dense(512, activation="relu")(d)

    prot_input = tf.keras.layers.Input(shape=(500,))
    emb = tf.keras.layers.Embedding(vocab_size, 40)(prot_input)
    conv1 = tf.keras.layers.Conv1D(64, 5, activation="relu")(emb)
    conv2 = tf.keras.layers.Conv1D(128, 5, activation="relu")(conv1)
    pool = tf.keras.layers.GlobalMaxPooling1D()(conv2)

    bp_input = tf.keras.layers.Input(shape=(bp_dim,))
    bp = tf.keras.layers.Dense(32, activation="relu")(bp_input)
    
    ngram_input = tf.keras.layers.Input(shape=(ngram_dim,))
    ng = tf.keras.layers.Dense(128, activation="relu")(ngram_input)

    z = tf.keras.layers.Concatenate()([d, pool, bp, ng])
    z = res_block(z, 512)
    z = se_block(z)
    z = tf.keras.layers.Dense(512, activation="relu")(z)
    z = tf.keras.layers.Dropout(DROPOUT_RATE)(z) 
    output = tf.keras.layers.Dense(1)(z)

    model = tf.keras.Model([drug_input, prot_input, bp_input, ngram_input], output)
    model.compile(optimizer=tf.keras.optimizers.AdamW(learning_rate=lr), loss=tf.keras.losses.Huber())
    return model

# =========================================================
# 🚀 QNN SINGLE FOLD EXECUTION (8:1:1 PROTOCOL)
# =========================================================

print(f"🚀 Starting Training under Cold-Start 8:1:1 Conditions with Optimized Bio-Parameters...")

with strategy.scope():
    model = build_qnn_model(
        X_smi_train.shape[1], 
        X_bp_train.shape[1], 
        X_ngram_train.shape[1], 
        vocab_size=41
    )

model.fit(
    [X_smi_train, X_prot_train, X_bp_train, X_ngram_train], y_train,
    validation_data=([X_smi_val, X_prot_val, X_bp_val, X_ngram_val], y_val),
    epochs=EPOCHS, 
    batch_size=BATCH_SIZE, 
    verbose=1
)

val_preds = model.predict([X_smi_val, X_prot_val, X_bp_val, X_ngram_val], verbose=0).flatten()
curr_test_preds = model.predict([X_smi_test, X_prot_test, X_bp_test, X_ngram_test], verbose=0).flatten()

test_results = {
    'mse': mse(y_test, curr_test_preds),
    'r2m': get_r2m(y_test, curr_test_preds),
    'ci': ci(y_test, curr_test_preds)
}

print(f"\n--- Final Test Results (8:1:1 Split) ---")
for metric, value in test_results.items():
    print(f"{metric.upper()}: {value:.3f}")
